# 14 — GraphBind nucleic-acid benchmark

The benchmark the field competes on: `DNA-129_Test`, `RNA-117_Test` (Xia et al., *NAR* 2021).

**The headline result of this notebook is already known, and it is negative.** Our DNA and RNA models were
trained on DevSet1010, and DevSet1010 **overlaps these test sets**. One protein is an exact match, three
more are substrings, and several others are >90% similar. Any number our current models produce here is
train-test contaminated and must not be reported.

This notebook exists to (a) make that finding reproducible, and (b) set up the only run that would be
legitimate: **retraining on `DNA-573_Train` / `RNA-495_Train`**.

| part | cost | purpose |
|---|---|---|
| 1. download + parse | ~1 min | verified against the published table |
| 2. **overlap audit** | ~2 min | **the gate — known to fail today** |
| 3. feasibility | seconds | can our pipeline run these chains |
| 4. retrain on their training set | hours + GPU | the only legitimate path |

In [ ]:
# ============================================================
# 0. Setup.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import sys, re, hashlib, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
EXT = BASE/'external_benchmarks'/'GraphBind'
EXT.mkdir(parents=True, exist_ok=True)
# This notebook imports NOTHING from the project — not pbind, not the GEAR backend. It reads the three
# label CSVs and the fold id files straight off Drive, so it runs in a fresh Colab with nothing set up.
CSV = {'DevSet1010':   BASE/'data'/'development_set'/'full_grouped_train_binding_sites_df.csv',
       'TestSet300':   BASE/'data'/'development_set'/'full_grouped_test_binding_sites_df.csv',
       'TestSetNew46': BASE/'data'/'independent_set'/'grouped_test_46_new_binding_sites.csv'}
FOLD_DIR = BASE/'data'/'development_set'          # ids_split1..5.txt
print('BASE', BASE, '| exists', BASE.exists())
for k, v in CSV.items():
    print(f'  {k:14s} {"co" if v.exists() else "THIEU"}  {v}')
print('  fold ids      ', 'co' if (FOLD_DIR/'ids_split1.txt').exists() else 'THIEU', ' ', FOLD_DIR)
print('EXT ', EXT)

## 1. Download and parse

`csbio.sjtu.edu.cn` serves a certificate for `*.sjtu.edu.cn`, which does not cover
`www.csbio.sjtu.edu.cn`, so HTTPS fails and the download must go over **HTTP**. The archive is 287 KB and
its SHA-256 is checked below, so the plaintext transport does not leave the content unverified.

Format, read off the actual files (the project page's description omits the sequence line):

```
test    >PDBID_Chain / SEQUENCE / LABELS                    3 lines per chain
train   >PDBID_Chain / SEQUENCE / TRANSFERRED / NON-TRANS    4 lines per chain
```

The paper's main tables use the **transferred** annotations, so that is what is parsed here.

In [ ]:
# ============================================================
# 1a. Download. HTTP only (see above); the SHA-256 is what makes it trustworthy.
# ============================================================
URL = 'http://www.csbio.sjtu.edu.cn/bioinf/GraphBind/files/Datasets.zip'
SHA256 = '0f32e4586a0bddcd2f1fac8bcd2bc53d9be7cf02093273d5b206e1a994e940f5'
zp = EXT/'Datasets.zip'

if not zp.exists():
    print('tai ve ...', URL)
    with urllib.request.urlopen(URL, timeout=120) as r:
        zp.write_bytes(r.read())
got = hashlib.sha256(zp.read_bytes()).hexdigest()
print(f'{zp.name}  {zp.stat().st_size/1e3:.0f} KB')
print(f'sha256 {got}')
assert got == SHA256, f'sha256 KHONG KHOP (cho doi {SHA256}) — xoa file va tai lai'
print('sha256 KHOP')

with zipfile.ZipFile(zp) as z:
    for n in z.namelist():
        if n.startswith('Datasets/') and n.endswith('.txt'):
            (EXT/Path(n).name).write_bytes(z.read(n))
print('giai nen ->', sorted(p.name for p in EXT.glob('*.txt')))

In [ ]:
# ============================================================
# 1b. Parse, and verify every count against the published Table 1.
# ============================================================
AA20 = set('ACDEFGHIKLMNPQRSTVWY')
PUBLISHED = {            # protein, positive, negative  (transferred annotations for the training sets)
    'DNA-129_Test':  (129,  2240,  35275),
    'RNA-117_Test':  (117,  2031,  35314),
    'DNA-573_Train': (573, 14479, 145404),
    'RNA-495_Train': (495, 14609, 122290),
}


def parse(fp, per):
    """per=3 test (name/seq/labels); per=4 train (name/seq/transferred/non-transferred)."""
    lines = [l.strip() for l in Path(fp).read_text().splitlines() if l.strip()]
    assert len(lines) % per == 0, f'{Path(fp).name}: {len(lines)} dong, khong chia het cho {per}'
    rows = []
    for i in range(0, len(lines), per):
        blk = lines[i:i + per]
        assert blk[0].startswith('>'), f'{Path(fp).name}: ban ghi {i//per} khong co header'
        seq, lab = blk[1], blk[2]                       # line 3 = transferred, the paper's main setting
        assert len(seq) == len(lab), f'{blk[0]}: seq {len(seq)} != nhan {len(lab)}'
        assert set(lab) <= {'0', '1'}, f'{blk[0]}: nhan khong phai 0/1'
        rows.append({'prot_id': blk[0][1:], 'sequence': seq,
                     'labels': np.frombuffer(lab.encode(), np.uint8) - ord('0')})
    return pd.DataFrame(rows)


BENCH = {}
for name, per in (('DNA-129_Test', 3), ('RNA-117_Test', 3), ('DNA-573_Train', 4), ('RNA-495_Train', 4)):
    df = parse(EXT/f'{name}.txt', per)
    BENCH[name] = df
    n, pos = len(df), int(sum(int(x.sum()) for x in df.labels))
    neg = int(df.sequence.str.len().sum()) - pos
    pn, pp, pg = PUBLISHED[name]
    ok = (n, pos, neg) == (pn, pp, pg)
    L = df.sequence.str.len()
    odd = sorted({c for s in df.sequence for c in s} - AA20)
    print(f'{name:16s} {n:4d} protein | {pos:6d} duong | {neg:7d} am | mat do {100*pos/(pos+neg):5.2f}% | '
          f'dai {L.min()}-{L.max()} | {"KHOP bang cong bo" if ok else f"LECH! cong bo {pn}/{pp}/{pg}"}')
    assert ok, f'{name}: so lieu khong khop bang cong bo — khong dung file nay'
    if odd:
        print(f'    ky tu ngoai 20 acid amin chuan: {odd}')

print()
print('Mat do duong cua ho ~5.4-6.0%; dich DNA cua minh tren TestSet300 chi 2.08%.')
print('AUPRC phu thuoc truc tiep vao mat do, nen so cua ho va so cua minh KHONG dat canh nhau duoc.')

## 2. Overlap audit — the gate

IDs cannot be compared (theirs are PDB+chain, ours are UniProt), so sequence is the only honest test.
Three checks, cheapest first:

1. **exact match** — the benchmark chain is verbatim one of our sequences;
2. **containment** — a PDB chain is usually a fragment of the UniProt entry, so this is the realistic leak;
3. **5-mer Jaccard screen** — catches obvious homologs.

GraphBind removed test chains >30% similar to **their own** training set. That says nothing about ours.

**Expected result: this fails.** Known contaminated pairs are listed in
`data/external_benchmarks/GraphBind/PROVENANCE.txt` in the repo.

In [ ]:
# ============================================================
# 2. Overlap audit.
# ============================================================
# DevSet1010 = the raw train CSV restricted to the official bindNode24 fold ids. The Drive copy of
# ids_split1..5.txt carries 1014 ids: the official 1010 plus 4 whose ESMFold structure does not reproduce
# the CSV sequence, which were never trained on.
EXCLUDED_DEV_IDS = {'C8BD48', 'P84801', 'Q00277', 'Q9NZV6'}
ids = []
for k in range(1, 6):
    ids += [x.strip() for x in (FOLD_DIR/f'ids_split{k}.txt').read_text().splitlines() if x.strip()]
official = [i for i in ids if i not in EXCLUDED_DEV_IDS]
assert len(official) == 1010, f'cho doi 1010 id chinh thuc, doc duoc {len(official)}'

S = {}
for name, fp in CSV.items():
    d = pd.read_csv(fp)
    d['prot_id'] = d.prot_id.astype(str)
    if name == 'DevSet1010':
        d = d[d.prot_id.isin(set(official))].reset_index(drop=True)
    S[name] = d
EXPECT = {'DevSet1010': 1010, 'TestSet300': 300, 'TestSetNew46': 46}
for name, d in S.items():
    assert len(d) == EXPECT[name], f'{name}: {len(d)} protein, cho doi {EXPECT[name]}'

OURS = [(str(r.prot_id), s, str(r.sequence)) for s, d in S.items() for r in d.itertuples()]
print(f'doi chieu voi {len(OURS)} protein cua minh ({ {s: len(d) for s, d in S.items()} })\n')

kmers = lambda s, k=5: {s[i:i + k] for i in range(len(s) - k + 1)}
OUR_K = [(p, s, q, kmers(q)) for p, s, q in OURS]
HITS = {}

for name in ('DNA-129_Test', 'RNA-117_Test'):
    exact, sub, near = [], [], []
    for r in BENCH[name].itertuples():
        bs, kb = str(r.sequence), kmers(str(r.sequence))
        best = (0.0, None, None)
        for pid, sp, osq, ok_ in OUR_K:
            if bs == osq:
                exact.append({'bench': r.prot_id, 'ours': pid, 'split': sp})
            elif len(bs) >= 20 and (bs in osq or osq in bs):
                sub.append({'bench': r.prot_id, 'ours': pid, 'split': sp,
                            'kieu': 'bench nam trong ours' if bs in osq else 'ours nam trong bench'})
            inter = len(kb & ok_)
            if inter:
                j = inter / len(kb | ok_)
                if j > best[0]:
                    best = (j, pid, sp)
        if best[0] >= 0.30:
            near.append({'bench': r.prot_id, 'ours': best[1], 'split': best[2], 'jaccard': round(best[0], 3)})
    HITS[name] = {'exact': exact, 'sub': sub, 'near': near}
    print(f'--- {name} ({len(BENCH[name])} protein) ---')
    print(f'    trung TUYET DOI       : {len(exact)}')
    print(f'    chuoi con cua nhau    : {len(sub)}')
    print(f'    5-mer Jaccard >= 0.30 : {len(near)}')
    for lbl, lst in (('TRUNG TUYET DOI', exact), ('CHUOI CON', sub), ('HO HANG', near)):
        if lst:
            d = pd.DataFrame(lst)
            if 'jaccard' in d:
                d = d.sort_values('jaccard', ascending=False)
            print(f'    [{lbl}]')
            print('     ' + d.head(15).to_string(index=False).replace('\n', '\n     '))
    print()

CLEAN = not any(h['exact'] or h['sub'] for h in HITS.values())
print('=' * 72)
if CLEAN:
    print('Khong thay trung. Van phai chay BLAST/CD-HIT o nguong 30% truoc khi cong bo.')
else:
    n = sum(len(h['exact']) + len(h['sub']) for h in HITS.values())
    print(f'CONG CHAN HONG: {n} protein test trung hoac nam trong tap huan luyen cua minh.')
    print('Model huan luyen tren DevSet1010 KHONG duoc cham tren hai tap test nay.')
    print('Duong hop le duy nhat: huan luyen lai tu dau tren DNA-573_Train / RNA-495_Train (phan 4).')
print('=' * 72)

## 3. Feasibility

Our pipeline needs an ESMFold structure plus ESM-2, ProtT5 and ESM-IF per chain. Hard limits come from
`pbind.inference.predictor`: 30–1000 residues, and 813 is the longest chain our models ever saw in
training. The ESM Atlas folding API refuses anything over ~400 residues, so longer chains need a local
ESMFold run on GPU.

In [ ]:
# ============================================================
# 3. How much of each set our pipeline can actually process.
# ============================================================
MIN_LEN, MAX_LEN = 30, 1000          # pbind.inference.predictor
MAX_TRAINED_LEN = 813                # longest chain our models were ever trained on
ESM_ATLAS_MAX = 400                  # the ESM Atlas folding API refuses longer sequences

rows = []
for name, df in BENCH.items():
    L = df.sequence.str.len().to_numpy()
    odd = np.array([not set(s) <= AA20 for s in df.sequence])
    rows.append({'bo': name, 'protein': len(df), 'dai_trung_vi': int(np.median(L)),
                 'ky_tu_la': int(odd.sum()),
                 f'<{MIN_LEN}': int((L < MIN_LEN).sum()),
                 f'>{MAX_LEN}': int((L > MAX_LEN).sum()),
                 f'>{MAX_TRAINED_LEN} ngoai pham vi train': int((L > MAX_TRAINED_LEN).sum()),
                 f'>{ESM_ATLAS_MAX} phai fold cuc bo': int((L > ESM_ATLAS_MAX).sum()),
                 'qua duoc API': int(((L >= MIN_LEN) & (L <= ESM_ATLAS_MAX) & ~odd).sum())})
print(pd.DataFrame(rows).to_string(index=False))
print()
print('Bo bot protein vi khong gap duoc = khong con so sanh duoc voi so da cong bo.')
print('Phai gap bang het, hoac khai ro tap con va khong goi do la ket qua tren benchmark.')

## 4. The only legitimate run: retrain on their training set

Training on `DNA-573_Train` / `RNA-495_Train` and testing on `DNA-129_Test` / `RNA-117_Test` removes the
contamination entirely, because DevSet1010 is never touched, and it is also the only way the comparison is
like-for-like with GraphBind, GraphSite, EquiPNAS and CrossBind.

Steps, each already implemented elsewhere in this project:

1. fold every training and test chain → `esmFold_pdb_files/`;
2. `pbind.features.registry.ensure_family` to build the feature families for the four new splits;
3. define **your own 5 folds** over the training set (the benchmark ships none) by sequence-identity
   clustering, and say so in the paper;
4. train V1 + MS hypergraphs on the new target, 3 seeds, reusing the LX-3 trainer from notebook 12;
5. pick ensemble weights and the threshold on the training folds only;
6. predict the test sets and put the AUPRC next to the published numbers.

Targets to beat — verify each against its own paper before citing:

| method | DNA-129_Test | RNA-117_Test |
|---|---|---|
| GraphBind (2021) | 0.497 | 0.204 |
| GraphSite (2021) | 0.522 | — |
| EquiPNAS (2024) | 0.569 | 0.320 |
| CrossBind (2023) | 0.628 | — |

One more check before step 1: confirm from the GraphBind paper how it defines a binding residue
(BioLiP's rule is ≥2 receptor-ligand atom pairs within the sum of van der Waals radii + 0.5 Å). If that
differs from how our labels were made, the targets are not the same task and nothing is comparable.

In [ ]:
# ============================================================
# 4. Folding queue for the retraining run. Runs only once part 2 is clean OR you are retraining from
#    scratch on their training set (in which case DevSet1010 contamination is irrelevant).
# ============================================================
RETRAIN_FROM_SCRATCH = False          # set True once you commit to the part-4 plan

assert CLEAN or RETRAIN_FROM_SCRATCH, (
    'Phan 2 chua sach va ban chua chon huan luyen lai tu dau. '
    'Dung lai: moi con so tren DNA-129_Test voi model hien tai deu bi nhiem.')

PDB_DIR = BASE/'esmFold_pdb_files'
jobs = []
for name, df in BENCH.items():
    for r in df.itertuples():
        L = len(r.sequence)
        if not (MIN_LEN <= L <= MAX_LEN) or not set(r.sequence) <= AA20:
            continue
        safe = re.sub(r'[^A-Za-z0-9_.-]', '_', str(r.prot_id))
        fp = PDB_DIR/f'GB_{name}__{safe}.pdb'
        if not fp.exists():
            jobs.append({'bo': name, 'prot_id': r.prot_id, 'len': L, 'out': str(fp),
                         'route': 'esm_atlas_api' if L <= ESM_ATLAS_MAX else 'esmfold_local_gpu'})
jq = pd.DataFrame(jobs)
jq.to_csv(EXT/'folding_jobs.csv', index=False)
print(f'{len(jq)} chain can gap' if len(jq) else 'da co cau truc cho tat ca')
if len(jq):
    print(jq.groupby(['bo', 'route']).size().to_string())
print('saved ->', EXT/'folding_jobs.csv')
print()
print('Gap bang dung ham ma notebook 09-13 dung, doc tu folding_jobs.csv —')
print('phai cung mot duong ong de cau truc giong het luc train.')